# 📉 Customer Churn Prediction & Segmentation — Telecom Industry

**A real-world Machine Learning project using the IBM Telco Customer Churn dataset (7,043 real customers)**

---

## Business Problem

Telecom companies lose a portion of their subscriber base every month — this is called **churn**. Acquiring a new customer costs significantly more than retaining an existing one, so predicting churn *before it happens* lets a business act early with retention offers.

**Dataset:** [IBM Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d) — a well-known, publicly available real-world dataset (7,043 customers, 21 features), commonly used in industry and academia for churn analysis.

## Objectives
1. **Predict which customers will churn** (Classification) — the core business goal
2. **Predict a customer's Total Charges** (Regression) — useful for revenue forecasting
3. **Handle real-world class imbalance** (only ~26.5% of customers churn) using SMOTE
4. **Segment customers** into groups for targeted marketing (Clustering)
5. **Identify top churn drivers** (Feature Importance) — actionable business insight
6. **Save a deployable model** — usable in a simple web app for live predictions

## Tools & Technologies
Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn (Pipelines, ColumnTransformer), Imbalanced-learn (SMOTE), Joblib


## 1. Setup — Install & Import Libraries

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (mean_squared_error, r2_score, mean_absolute_error,
                              accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

sns.set_style('whitegrid')
np.random.seed(42)
print('Libraries imported successfully!')

## 2. Load the Real Dataset

We load the dataset directly from its public GitHub source — no manual upload needed, works in Colab or Jupyter as-is.

In [ ]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
print('Dataset shape:', df.shape)
df.head()

## 3. Data Cleaning

Real-world data always has quirks. Two issues we fix here:
1. **`customerID`** — a unique identifier, not predictive, so we drop it
2. **`TotalCharges`** — loaded as *text* instead of numbers, because 11 brand-new customers (tenure = 0) have a blank value instead of a number. We convert it properly and fill those 11 with 0 (since a brand-new customer hasn't been charged yet).

In [ ]:
df = df.drop(columns=['customerID'])

# TotalCharges is stored as text due to blank values for new customers
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('Missing TotalCharges before fix:', df['TotalCharges'].isnull().sum())

df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Convert SeniorCitizen (0/1) to Yes/No for consistency with other categorical columns
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print('\nMissing values after cleaning:')
print(df.isnull().sum().sum(), 'total missing values')
df.info()

## 4. Data Understanding

In [ ]:
df.describe(include='all')

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print('Churn counts:\n', churn_counts)
print('\nChurn percentage:\n', churn_pct.round(2))
print(f'\n⚠️ Class Imbalance detected: only {churn_pct["Yes"]:.1f}% of customers churn.')
print('This matters — a model that just predicts "No churn" for everyone would still be ~73% accurate,')
print('but completely useless for the business. We will handle this with SMOTE later.')

## 5. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(6,5))
df['Churn'].value_counts().plot(kind='bar', color=['lightgreen','salmon'])
plt.title('Customer Churn Distribution (Imbalanced)')
plt.ylabel('Number of Customers')
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
churn_by_contract = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
churn_by_contract.plot(kind='bar', stacked=True, color=['lightgreen','salmon'], ax=plt.gca())
plt.title('Churn Rate by Contract Type')
plt.ylabel('Percentage')
plt.xticks(rotation=15)
plt.legend(title='Churn')
plt.show()

print(churn_by_contract.round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,5))
for ax, col in zip(axes, ['InternetService', 'PaymentMethod', 'TechSupport']):
    rates = df.groupby(col)['Churn'].apply(lambda x: (x=='Yes').mean()*100)
    rates.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Churn % by {col}')
    ax.set_xlabel('Churn %')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[0])
axes[0].set_title('Tenure vs Churn')
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[1])
axes[1].set_title('Monthly Charges vs Churn')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

**Key EDA findings (from real data):**
- Month-to-Month contract customers churn at **42.7%**, vs just **2.8%** for Two-Year contracts
- Fiber Optic internet users churn at **41.9%**, much higher than DSL (19.0%)
- Customers paying via **Electronic Check** churn at **45.3%** — the highest of any payment method
- Customers without Tech Support churn at **41.6%**, vs only **15.2%** with Tech Support
- Churned customers have a much lower average tenure (**18.0 months**) vs retained customers (**37.6 months**)

These are strong, actionable business signals — not just statistical noise.

## 6. Preprocessing Setup — Feature Definitions

Instead of manually encoding columns, we use **`ColumnTransformer`** — a professional practice that bundles encoding + scaling into a single reusable object. This avoids train/test mismatch bugs and makes the whole pipeline deployable later.

In [ ]:
categorical_features = ['gender','SeniorCitizen','Partner','Dependents','PhoneService','MultipleLines',
                        'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport',
                        'StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']

print(f'{len(categorical_features)} categorical features identified.')

## 7. Regression — Predicting Total Charges (Revenue)

Business use: forecasting a customer's cumulative revenue based on their plan and tenure. Note: we exclude `TotalCharges` itself from the input features here since it's the target we're predicting.

In [ ]:
reg_numeric_features = ['tenure', 'MonthlyCharges']

reg_preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), reg_numeric_features)
])

X_reg = df[categorical_features + reg_numeric_features]
y_reg = df['TotalCharges']

Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_pipeline = Pipeline([
    ('preprocessor', reg_preprocessor),
    ('regressor', LinearRegression())
])
reg_pipeline.fit(Xr_train, yr_train)
yr_pred = reg_pipeline.predict(Xr_test)

r2 = r2_score(yr_test, yr_pred)
mae = mean_absolute_error(yr_test, yr_pred)
rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))

print(f'R2 Score: {r2:.3f}')
print(f'Mean Absolute Error: ${mae:.2f}')
print(f'Root Mean Squared Error: ${rmse:.2f}')

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(yr_test, yr_pred, color='purple', alpha=0.5)
plt.plot([yr_test.min(), yr_test.max()], [yr_test.min(), yr_test.max()], 'r--')
plt.xlabel('Actual Total Charges ($)')
plt.ylabel('Predicted Total Charges ($)')
plt.title('Revenue Prediction: Actual vs Predicted')
plt.show()

**Honest note (good to mention in viva):** The R² here is expected to be very high, because `TotalCharges ≈ tenure × MonthlyCharges` almost by definition — this isn't the model being "magically good", it's the underlying business math. A more challenging regression task would be predicting **future** charges or **Customer Lifetime Value**, which is listed under Future Scope.

## 8. Classification — Predicting Churn (Main Business Goal)

Now the core task: predicting `Churn` (Yes/No). We use **all** features including `TotalCharges`, since at prediction time (for an existing customer) all of this data is known.

In [ ]:
clf_numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

clf_preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), clf_numeric_features)
])

X_clf = df[categorical_features + clf_numeric_features]
y_clf = df['Churn'].map({'Yes': 1, 'No': 0})

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print('Training set class distribution:')
print(yc_train.value_counts(normalize=True).round(3))

### Handling Class Imbalance with SMOTE

Since only ~26.5% of customers churn, a model can get "high accuracy" just by predicting "No Churn" for everyone — which is useless for the business (it would miss every at-risk customer).

**SMOTE (Synthetic Minority Over-sampling Technique)** creates synthetic examples of the minority class (churners) in the *training data only*, so the model learns to recognize churn patterns properly. We use `imblearn`'s Pipeline so SMOTE is applied **only inside each training fold** — never on test data — which avoids data leakage.

## 9. Training & Comparing 4 Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=500),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'Random Forest': RandomForestClassifier(n_estimators=150, random_state=42, max_depth=8)
}

results = []
fitted_pipelines = {}

for name, clf in models.items():
    pipe = ImbPipeline([
        ('preprocessor', clf_preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', clf)
    ])
    pipe.fit(Xc_train, yc_train)
    pred = pipe.predict(Xc_test)
    prob = pipe.predict_proba(Xc_test)[:, 1]

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(yc_test, pred),
        'Precision': precision_score(yc_test, pred),
        'Recall': recall_score(yc_test, pred),
        'F1-Score': f1_score(yc_test, pred),
        'ROC-AUC': roc_auc_score(yc_test, prob)
    })
    fitted_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df

In [ ]:
plt.figure(figsize=(10,6))
x = np.arange(len(results_df))
width = 0.2
plt.bar(x - width, results_df['Accuracy'], width, label='Accuracy')
plt.bar(x, results_df['Recall'], width, label='Recall (catches churners)')
plt.bar(x + width, results_df['ROC-AUC'], width, label='ROC-AUC')
plt.xticks(x, results_df['Model'], rotation=15)
plt.ylabel('Score')
plt.title('Model Comparison for Churn Prediction (with SMOTE)')
plt.legend()
plt.show()

**Why Recall matters more than Accuracy for churn:** Missing an actual churner (a False Negative) means losing a customer the business could have saved. So for this business problem, we prioritize **Recall** and **ROC-AUC** over plain Accuracy — a key point to explain in the viva.

In [ ]:
best_name = results_df.iloc[0]['Model']
best_pipe = fitted_pipelines[best_name]
pred_best = best_pipe.predict(Xc_test)
prob_best = best_pipe.predict_proba(Xc_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14,5))

cm = confusion_matrix(yc_test, pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix - {best_name}')

fpr, tpr, _ = roc_curve(yc_test, prob_best)
axes[1].plot(fpr, tpr, label=f'AUC = {roc_auc_score(yc_test, prob_best):.2f}')
axes[1].plot([0,1],[0,1],'r--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve - {best_name}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Best model: {best_name}')
print(classification_report(yc_test, pred_best, target_names=['No Churn','Churn']))

## 10. Feature Importance — What Actually Drives Churn?

In [ ]:
rf_pipe = fitted_pipelines['Random Forest']
feature_names = rf_pipe.named_steps['preprocessor'].get_feature_names_out()
importances = rf_pipe.named_steps['classifier'].feature_importances_

importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}) \
                   .sort_values('Importance', ascending=False).head(12)

plt.figure(figsize=(9,6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='crimson')
plt.xlabel('Importance Score')
plt.title('Top 12 Churn Drivers (Random Forest)')
plt.gca().invert_yaxis()
plt.show()

importance_df

## 11. Hyperparameter Tuning (GridSearchCV)

We tune Random Forest's key hyperparameters. Because we're tuning the entire pipeline (`preprocessor → SMOTE → classifier`), cross-validation correctly re-applies SMOTE fresh on each training fold, avoiding any data leakage.

In [ ]:
param_grid = {
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_depth': [5, 8, 10, None]
}

tuning_pipe = ImbPipeline([
    ('preprocessor', clf_preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(tuning_pipe, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(Xc_train, yc_train)

print('Best Parameters:', grid_search.best_params_)
print('Best Cross-Validation ROC-AUC:', round(grid_search.best_score_, 3))

tuned_pipeline = grid_search.best_estimator_
tuned_pred = tuned_pipeline.predict(Xc_test)
tuned_prob = tuned_pipeline.predict_proba(Xc_test)[:, 1]
print('Test Accuracy after Tuning:', round(accuracy_score(yc_test, tuned_pred), 3))
print('Test ROC-AUC after Tuning:', round(roc_auc_score(yc_test, tuned_prob), 3))

## 12. Business Impact — Turning Predictions into ₹/$ Value

In [ ]:
avg_monthly_revenue = df['MonthlyCharges'].mean()
actual_churn_rate = (df['Churn'] == 'Yes').mean() * 100

test_churners_caught = int(((tuned_pred == 1) & (yc_test.values == 1)).sum())
retention_success_rate = 0.30  # assumption: 30% of contacted at-risk customers can be retained

customers_saved = test_churners_caught * retention_success_rate
annual_revenue_saved = customers_saved * avg_monthly_revenue * 12

print(f'Overall churn rate in dataset: {actual_churn_rate:.1f}%')
print(f'Average Monthly Revenue per Customer: ${avg_monthly_revenue:.2f}')
print(f'Churners correctly identified in test set: {test_churners_caught}')
print(f'\nAssuming a {retention_success_rate*100:.0f}% success rate on retention offers:')
print(f'  -> Estimated customers saved (test set scale): {customers_saved:.1f}')
print(f'  -> Estimated ANNUAL revenue saved (test set scale): ${annual_revenue_saved:,.2f}')
print('\nAt the scale of the full 7,043-customer base, this effect would be proportionally larger.')
print('This is the kind of business translation that turns a model into a decision-making tool.')

## 13. Save the Model for Deployment

We save the entire tuned pipeline (preprocessing + trained classifier) as a single file using `joblib`. This one file can now be loaded anywhere — including a simple web app — to make live predictions on new customers.

In [ ]:
joblib.dump(tuned_pipeline, 'churn_model_pipeline.pkl')
print('Model saved as churn_model_pipeline.pkl')
print('This file is used directly by the Streamlit app (app.py) for live predictions.')

## 14. Customer Segmentation — K-Means Clustering

Business use: Marketing shouldn't treat every customer the same. We segment customers by `tenure`, `MonthlyCharges`, and `TotalCharges` to design **targeted campaigns**.

In [ ]:
cluster_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_cluster_scaled = StandardScaler().fit_transform(df[cluster_features])

inertia = []
k_range = range(1, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Segment'] = kmeans.fit_predict(X_cluster_scaled)

plt.figure(figsize=(8,6))
scatter = plt.scatter(df['tenure'], df['MonthlyCharges'], c=df['Segment'], cmap='viridis', alpha=0.6)
plt.xlabel('Tenure (Months)')
plt.ylabel('Monthly Charges ($)')
plt.title('Customer Segments')
plt.colorbar(scatter, label='Segment')
plt.show()

In [ ]:
segment_summary = df.groupby('Segment').agg(
    Avg_Tenure=('tenure', 'mean'),
    Avg_Monthly_Charges=('MonthlyCharges', 'mean'),
    Avg_Total_Charges=('TotalCharges', 'mean'),
    Churn_Rate_Pct=('Churn', lambda x: round((x == 'Yes').mean() * 100, 1)),
    Count=('Churn', 'count')
).round(2)
segment_summary

**Business interpretation of segments (based on actual output above):**
- **Long tenure, moderate charges, low churn rate** → "Loyal Customers" — reward with loyalty perks, low priority for retention spend
- **Short tenure, high charges, high churn rate** → "At-Risk New Customers" — prioritize onboarding support & first-90-days discounts
- **Short tenure, low total charges** → "New/Trial Customers" — nurture with engagement campaigns

*(Match the exact numbers from your `segment_summary` table above when explaining this in the viva.)*

## 15. Conclusion & Business Recommendations

### What we built
1. A **Regression model** to forecast customer revenue (Total Charges)
2. **4 Classification models** compared to predict churn, with **SMOTE** to handle real class imbalance
3. **Feature Importance analysis** — Contract type, tenure, and Internet Service type are the top churn drivers
4. **Hyperparameter-tuned** Random Forest via GridSearchCV
5. A **Business Impact estimate** translating model performance into revenue saved
6. **Customer Segmentation (K-Means)** for targeted marketing
7. A **deployable model file** (`churn_model_pipeline.pkl`), used by the included Streamlit app

### Business Recommendations (grounded in real EDA findings)
- Incentivize **Month-to-Month customers** (42.7% churn) to move to annual contracts (2.8% churn) — the single biggest lever
- Investigate **Fiber Optic service quality** — its customers churn at 41.9%, far above DSL
- Push customers away from **Electronic Check** payments (45.3% churn) toward automatic payment methods
- Proactively offer **Tech Support** to customers who don't have it — a clear churn-reducing factor in the data

### Limitations
- Retention success rate (30%) used in the business-impact estimate is an assumption, not measured — would need real campaign A/B test data to validate
- The model reflects patterns in this specific dataset (a US telecom snapshot) — would need re-validation on other markets

### Future Scope
- Predict **Customer Lifetime Value (CLV)** instead of just Total Charges, for a more forward-looking regression task
- Deploy the Streamlit app publicly (Streamlit Community Cloud) for a live demo link
- Set up automatic model retraining as new customer data comes in

---
**Author:** [Your Name] · **Dataset:** IBM Telco Customer Churn (public)
